# Purpose:
- Run GLM on thyme natural images sessions
    - From v00 (without pupil)
- Changed from separate design matrix generation and GLM fitting to the same capsule
    - Easier to run the whole series of sessions
    - When the same design matrix is necessary, I can use the results from this one.
        - When capsularized, I can track where the design matrix came from.

# Summary:
- Added validation for checking NaN values in W
- Checked (again) how multiple planes can have the shared X (due to interpolation to stimulus_presentations timestamps)
- Confirmed that dask works a lot faster (18 min vs 130 min) (about 14%, or 86% reduction, or roughly 1/7)
- Dask results the same as looping (even with all the warning messages)

# Update:
- Added pupil: dm version 01 (01/24/2025)
- Added DFF 01/24/2025
- Fit using dm version 02 (updated kernel lengths)

In [1]:
import sys
sys.path.append('/root/capsule/code/')

import os
import glob
from pathlib import Path
import numpy as np
import xarray as xr
import json
from matplotlib import pyplot as plt
from pympler import asizeof

import glm_fit_tools as gft

# notebook dev
%load_ext autoreload
%autoreload 2
%matplotlib inline


# Run all Thyme natural images sessions

In [2]:
def get_prop_support(response_trim):
    prop_support = []
    for cri in response_trim.cell_roi_id.values:
        trace = response_trim.sel(cell_roi_id=cri)
        prop_support.append(len(np.where(trace)[0]) / len(trace))
    prop_support = np.array(prop_support)
    return prop_support


def filter_response_matrix(response, prop_support_threshold=0.01):
    prop_support = get_prop_support(response)
    filtered_inds = np.where(prop_support >= prop_support_threshold)[0]
    response_filtered = response.isel(cell_roi_id=filtered_inds)
    return response_filtered
    
    

In [3]:
data_type = 'events'
dm_version = 2
load_path = Path('/root/capsule/scratch/Thyme')
save_dir = load_path

data_dir = '/root/capsule/data'
data_folders = [d for d in glob.glob(data_dir + '/*') if Path(d).is_dir()]
raw_paths = np.sort([d for d in data_folders if ('processed' not in d.split('/')[-1]) and
                    ('dlc-eye' not in d.split('/')[-1]) and
                    ('multiplane-ophys' in d.split('/')[-1]) and 
                    ('stimuli' not in d.split('/')[-1]) and
                    ('stim-response' not in d.split('/')[-1]) and
                    ('ROICat' not in d.split('/')[-1])])
session_names = [d.split('/')[-1] for d in raw_paths]
session_names

['multiplane-ophys_736963_2024-07-24_08-49-57',
 'multiplane-ophys_736963_2024-07-26_09-37-45',
 'multiplane-ophys_736963_2024-07-29_09-00-58',
 'multiplane-ophys_736963_2024-07-30_09-11-03',
 'multiplane-ophys_736963_2024-08-01_08-59-00',
 'multiplane-ophys_736963_2024-08-05_09-19-25',
 'multiplane-ophys_736963_2024-08-06_08-52-03',
 'multiplane-ophys_736963_2024-08-07_09-11-10',
 'multiplane-ophys_736963_2024-08-09_08-58-36',
 'multiplane-ophys_736963_2024-08-12_09-17-19',
 'multiplane-ophys_736963_2024-08-13_08-57-29']

In [4]:
run_params

NameError: name 'run_params' is not defined

In [5]:
for session_name in session_names[6:]:
    save_fn = save_dir / f'glm_results_v{dm_version:02}_{session_name}_{data_type}.npy'
    # save_fn = save_dir / f'glm_results_v{dm_version:02}_{session_name}_{data_type}_00.npy'
    if save_fn.exists():
        print(f'\n{session_name} already processed\nSkipping...\n\n\n')
    else:
        print(f'Processing {session_name}')
        X, response, response_info, run_params, unstd_features, use_indices = \
            gft.load_data(session_name, data_type, dm_version, load_path=load_path)

        print('-------------------------------')
        print('Data loaded.')
        print('-------------------------------\n')
        
        ophys_frame_rate = response_info['ophys_frame_rate']
        # trim X and response based on the shift
        X_trim = X[use_indices, :]
        response_trim = response[use_indices, :]

        # Filter out cells based on prop event frames (<1%)
        response_trim_filtered = filter_response_matrix(response_trim)
        # TODO: this must be included in gft or dmt
        
        fit_params = gft.default_fit_params()
        # TODO: stratification parameters (threshold) can be improved
        stratified_list = gft.set_stratified_list(fit_params, X, unstd_features, use_indices, ophys_frame_rate)
        stratified_frames, cv_inds_stratified = gft.get_stratified_folds(fit_params, stratified_list)
        
        lambdas_cv, W_cv, var_explained_train_cv, var_explained_test_cv, ve_test_train_ratio_cv = \
                gft.collect_session_results(run_params, fit_params, X_trim,
                                            response_trim_filtered, stratified_frames, cv_inds_stratified,
                                            parallel=True)
        print('-------------------------------')
        print('Model fit.')
        print('-------------------------------\n')
        # TODO: examine how to suppress dask warnings
        
        var_explained_mean_model = \
            gft.get_full_session_var_explained_from_mean_model(W_cv, X_trim, response_trim_filtered)
        
        gft.save_glm_results(dm_version, session_name, data_type, 
                        fit_params,  use_indices, stratified_frames, cv_inds_stratified,
                        lambdas_cv, W_cv,
                        var_explained_train_cv, var_explained_test_cv, ve_test_train_ratio_cv, 
                        var_explained_mean_model,
                        save_dir=save_dir)
        
        print('-------------------------------')
        print('Results saved.')
        print('-------------------------------\n')


multiplane-ophys_736963_2024-08-06_08-52-03 already processed
Skipping...




multiplane-ophys_736963_2024-08-07_09-11-10 already processed
Skipping...



Processing multiplane-ophys_736963_2024-08-09_08-58-36
-------------------------------
Data loaded.
-------------------------------



2025-01-31 06:11:52,565 - distributed.worker - ERROR - Failed to communicate with scheduler during heartbeat.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.9/site-packages/distributed/comm/tcp.py", line 225, in read
    frames_nosplit_nbytes_bin = await stream.read_bytes(fmt_size)
tornado.iostream.StreamClosedError: Stream is closed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/opt/conda/lib/python3.9/site-packages/distributed/worker.py", line 1250, in heartbeat
    response = await retry_operation(
  File "/opt/conda/lib/python3.9/site-packages/distributed/utils_comm.py", line 459, in retry_operation
    return await retry(
  File "/opt/conda/lib/python3.9/site-packages/distributed/utils_comm.py", line 438, in retry
    return await coro()
  File "/opt/conda/lib/python3.9/site-packages/distributed/core.py", line 1254, in send_recv_from_rpc
    return await send_recv(comm=comm, op=key, **kwargs)
  

-------------------------------
Model fit.
-------------------------------

-------------------------------
Results saved.
-------------------------------

Processing multiplane-ophys_736963_2024-08-12_09-17-19
-------------------------------
Data loaded.
-------------------------------

-------------------------------
Model fit.
-------------------------------

-------------------------------
Results saved.
-------------------------------

Processing multiplane-ophys_736963_2024-08-13_08-57-29
-------------------------------
Data loaded.
-------------------------------

-------------------------------
Model fit.
-------------------------------

-------------------------------
Results saved.
-------------------------------



In [6]:
for session_name in session_names[:6]:
    save_fn = save_dir / f'glm_results_v{dm_version:02}_{session_name}_{data_type}.npy'
    # save_fn = save_dir / f'glm_results_v{dm_version:02}_{session_name}_{data_type}_00.npy'
    if save_fn.exists():
        print(f'\n{session_name} already processed\nSkipping...\n\n\n')
    else:
        print(f'Processing {session_name}')
        X, response, response_info, run_params, unstd_features, use_indices = \
            gft.load_data(session_name, data_type, dm_version, load_path=load_path)

        print('-------------------------------')
        print('Data loaded.')
        print('-------------------------------\n')
        
        ophys_frame_rate = response_info['ophys_frame_rate']
        # trim X and response based on the shift
        X_trim = X[use_indices, :]
        response_trim = response[use_indices, :]

        # Filter out cells based on prop event frames (<1%)
        response_trim_filtered = filter_response_matrix(response_trim)
        # TODO: this must be included in gft or dmt
        
        fit_params = gft.default_fit_params()
        # TODO: stratification parameters (threshold) can be improved
        stratified_list = gft.set_stratified_list(fit_params, X, unstd_features, use_indices, ophys_frame_rate)
        stratified_frames, cv_inds_stratified = gft.get_stratified_folds(fit_params, stratified_list)
        
        lambdas_cv, W_cv, var_explained_train_cv, var_explained_test_cv, ve_test_train_ratio_cv = \
                gft.collect_session_results(run_params, fit_params, X_trim,
                                            response_trim_filtered, stratified_frames, cv_inds_stratified,
                                            parallel=True)
        print('-------------------------------')
        print('Model fit.')
        print('-------------------------------\n')
        # TODO: examine how to suppress dask warnings
        
        var_explained_mean_model = \
            gft.get_full_session_var_explained_from_mean_model(W_cv, X_trim, response_trim_filtered)
        
        gft.save_glm_results(dm_version, session_name, data_type, 
                        fit_params,  use_indices, stratified_frames, cv_inds_stratified,
                        lambdas_cv, W_cv,
                        var_explained_train_cv, var_explained_test_cv, ve_test_train_ratio_cv, 
                        var_explained_mean_model,
                        save_dir=save_dir)
        
        print('-------------------------------')
        print('Results saved.')
        print('-------------------------------\n')

Processing multiplane-ophys_736963_2024-07-24_08-49-57
-------------------------------
Data loaded.
-------------------------------

-------------------------------
Model fit.
-------------------------------

-------------------------------
Results saved.
-------------------------------

Processing multiplane-ophys_736963_2024-07-26_09-37-45
-------------------------------
Data loaded.
-------------------------------

-------------------------------
Model fit.
-------------------------------

-------------------------------
Results saved.
-------------------------------

Processing multiplane-ophys_736963_2024-07-29_09-00-58
-------------------------------
Data loaded.
-------------------------------



2025-01-31 09:26:37,869 - distributed.worker - ERROR - Failed to communicate with scheduler during heartbeat.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.9/site-packages/distributed/comm/tcp.py", line 225, in read
    frames_nosplit_nbytes_bin = await stream.read_bytes(fmt_size)
tornado.iostream.StreamClosedError: Stream is closed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/opt/conda/lib/python3.9/site-packages/distributed/worker.py", line 1250, in heartbeat
    response = await retry_operation(
  File "/opt/conda/lib/python3.9/site-packages/distributed/utils_comm.py", line 459, in retry_operation
    return await retry(
  File "/opt/conda/lib/python3.9/site-packages/distributed/utils_comm.py", line 438, in retry
    return await coro()
  File "/opt/conda/lib/python3.9/site-packages/distributed/core.py", line 1254, in send_recv_from_rpc
    return await send_recv(comm=comm, op=key, **kwargs)
  

-------------------------------
Model fit.
-------------------------------

-------------------------------
Results saved.
-------------------------------

Processing multiplane-ophys_736963_2024-07-30_09-11-03
-------------------------------
Data loaded.
-------------------------------



2025-01-31 09:55:04,465 - distributed.worker - ERROR - Failed to communicate with scheduler during heartbeat.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.9/site-packages/distributed/comm/tcp.py", line 225, in read
    frames_nosplit_nbytes_bin = await stream.read_bytes(fmt_size)
tornado.iostream.StreamClosedError: Stream is closed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/opt/conda/lib/python3.9/site-packages/distributed/worker.py", line 1250, in heartbeat
    response = await retry_operation(
  File "/opt/conda/lib/python3.9/site-packages/distributed/utils_comm.py", line 459, in retry_operation
    return await retry(
  File "/opt/conda/lib/python3.9/site-packages/distributed/utils_comm.py", line 438, in retry
    return await coro()
  File "/opt/conda/lib/python3.9/site-packages/distributed/core.py", line 1254, in send_recv_from_rpc
    return await send_recv(comm=comm, op=key, **kwargs)
  

-------------------------------
Model fit.
-------------------------------

-------------------------------
Results saved.
-------------------------------

Processing multiplane-ophys_736963_2024-08-01_08-59-00
-------------------------------
Data loaded.
-------------------------------

-------------------------------
Model fit.
-------------------------------

-------------------------------
Results saved.
-------------------------------

Processing multiplane-ophys_736963_2024-08-05_09-19-25
-------------------------------
Data loaded.
-------------------------------

-------------------------------
Model fit.
-------------------------------

-------------------------------
Results saved.
-------------------------------



# Testing with one example

In [9]:

data_dir = '/root/capsule/data'
data_folders = [d for d in glob.glob(data_dir + '/*') if Path(d).is_dir()]
raw_paths = np.sort([d for d in data_folders if ('processed' not in d.split('/')[-1]) and
                 ('dlc-eye' not in d.split('/')[-1]) and
                 ('multiplane-ophys' in d.split('/')[-1]) and 
                 ('stimuli' not in d.split('/')[-1])])
session_names = [d.split('/')[-1] for d in raw_paths]



In [10]:
session_names

['multiplane-ophys_736963_2024-07-24_08-49-57',
 'multiplane-ophys_736963_2024-07-26_09-37-45',
 'multiplane-ophys_736963_2024-07-29_09-00-58',
 'multiplane-ophys_736963_2024-07-30_09-11-03',
 'multiplane-ophys_736963_2024-08-01_08-59-00',
 'multiplane-ophys_736963_2024-08-05_09-19-25',
 'multiplane-ophys_736963_2024-08-06_08-52-03',
 'multiplane-ophys_736963_2024-08-07_09-11-10',
 'multiplane-ophys_736963_2024-08-09_08-58-36',
 'multiplane-ophys_736963_2024-08-12_09-17-19',
 'multiplane-ophys_736963_2024-08-13_08-57-29']

In [11]:
session_name = session_names[0]
data_type = 'events'
dm_version = 0
load_path = Path('/root/capsule/scratch/Thyme')

X, response, response_info, run_params, unstd_features, use_indices = \
    gft.load_data(session_name, data_type, dm_version, load_path=load_path)

# trim X and response based on the shift
X_trim = X[use_indices, :]
response_trim = response[use_indices, :]

# Filter out cells based on prop event frames (<1%)

ophys_frame_rate = response_info['ophys_frame_rate']

In [31]:
prop_support = get_prop_support(response_trim)
prop_support_threshold = 0.01
filtered_inds = np.where(prop_support >= prop_support_threshold)[0]
response_trim_filtered = response_trim.isel(cell_roi_id=filtered_inds)

In [35]:
fit_params = gft.default_fit_params()
stratified_list = gft.set_stratified_list(fit_params, X, unstd_features, use_indices, ophys_frame_rate)
stratified_frames, cv_inds_stratified = gft.get_stratified_folds(fit_params, stratified_list)

In [69]:
lambdas_cv, W_cv, var_ratio_train_cv, var_ratio_test_cv, vr_test_train_ratio_cv =\
    gft.collect_session_results(run_params, fit_params, X_trim, response_trim_filtered, stratified_frames, cv_inds_stratified,
                                parallel=False)

In [73]:
save_fn = '/root/capsule/scratch/temp/W_cv_noparallel_736963_2024-07-24.npy'
np.save(save_fn, W_cv)

In [74]:
loaded_W_cv = np.load(save_fn)

# Compute time
- 130 min for multiplane-ophys_736963_2024-07-24_08-49-57
- 18 min with dask.

In [78]:
lambdas_cv_par, W_cv_par, var_ratio_train_cv_par, var_ratio_test_cv_par, vr_test_train_ratio_cv_par =\
    gft.collect_session_results(run_params, fit_params, X_trim, response_trim_filtered, stratified_frames, cv_inds_stratified,
                                parallel=True)

/opt/conda/lib/python3.9/site-packages/distributed/client.py:3362: UserWarning: Sending large graph of size 187.35 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/opt/conda/lib/python3.9/site-packages/distributed/client.py:3362: UserWarning: Sending large graph of size 187.35 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/opt/conda/lib/python3.9/site-packages/distributed/client.py:3362: UserWarning: Sending large graph of size 187.35 MiB.
This may cause some slowdown.
Consider loading the data with Dask dire

# Compare the model performances between dask and not

In [240]:
var_ratio_mean_model_single = gft.get_full_session_var_ratio_from_mean_model(W_cv, X_trim, response_trim_filtered)

In [241]:
var_ratio_mean_model_par = gft.get_full_session_var_ratio_from_mean_model(W_cv_par, X_trim, response_trim_filtered)

In [245]:
np.array_equal(var_ratio_mean_model_single, var_ratio_mean_model_par)

True

# Test saving

In [265]:

gft.save_glm_restuls(dm_version, session_name, data_type, 
                     fit_params,  use_indices, stratified_frames, cv_inds_stratified,
                     lambdas_cv, W_cv,
                     var_ratio_train_cv, var_ratio_test_cv,
                     vr_test_train_ratio_cv, var_ratio_mean_model,
                     save_dir=load_path)



/root/capsule/scratch/Thyme/glm_results_v00_multiplane-ophys_736963_2024-07-24_08-49-57_events.npy exists!

Adding_suffix...
New save filename  = /root/capsule/scratch/Thyme/glm_results_v00_multiplane-ophys_736963_2024-07-24_08-49-57_events_01.npy


PosixPath('/root/capsule/scratch/Thyme/glm_results_v00_multiplane-ophys_736963_2024-07-24_08-49-57_events_01.npy')

In [ ]:
save_dir_base = '/root/capsule/scratch'
save_folder_name = 'Thyme'
save_dir = Path(save_dir_base ) / folder_name


In [ ]:
save_fn = Path(save_fn)
    save_fn.parent.mkdir(parents=True, exist_ok=True)

# NaN issue

In [82]:
len(np.where(np.isnan(loaded_W_cv))[0])

7146925

In [80]:
len(np.where(np.isnan(W_cv_par))[0])

7146925

In [84]:
for i in range(4):
    print(np.all(np.where(np.isnan(W_cv))[i] == np.where(np.isnan(W_cv_par))[i]))

True
True
True
True


### All the indices are the same.
- Not related to warning messages from dask
- Could be due to random stratification

In [86]:
W_cv

<xarray.DataArray (test_fold_ind: 5, model: 32, weights: 151, cell_roi_id: 635)> Size: 123MB
array([[[[-3.21800146e-03,  1.75650484e-03,  1.94503345e-03, ...,
           2.96901546e-03,  1.36375263e-02,  5.26165594e-03],
         [ 4.66368199e-03,  1.00470187e-02,  5.05241984e-02, ...,
           1.73778314e-02,  2.16767038e-02,  1.74733317e-02],
         [-2.68123246e-03,  1.56128734e-03,  1.65456802e-02, ...,
          -1.59728474e-03, -1.94020901e-02, -5.01453608e-03],
         ...,
         [ 6.47890033e-04,  4.43339906e-04,  5.38367469e-04, ...,
          -2.29575859e-04,  1.40724789e-03, -1.12411516e-03],
         [ 2.61901640e-04, -3.65061820e-04,  1.17177388e-04, ...,
           6.67060688e-05,  8.59074741e-04,  8.84517873e-04],
         [ 1.32429972e-03,  2.56214715e-04,  7.36689530e-05, ...,
           1.06142872e-04, -2.92492354e-03, -6.46081711e-04]],

        [[-4.27402503e-03,  1.74545466e-03,  1.91891589e-03, ...,
           2.96211398e-03,  1.64256469e-02,  5.24649519e-03],
         [ 7.51206766e-03,  1.01064806e-02,  5.05944705e-02, ...,
           1.73980991e-02,  2.71727590e-02,  1.75399026e-02],
         [-4.84691756e-03,  1.49186409e-03,  1.64382400e-02, ...,
          -1.62692504e-03, -2.66077960e-02, -5.09704340e-03],
...
         [            nan,             nan,             nan, ...,
                      nan,             nan,             nan],
         [            nan,             nan,             nan, ...,
                      nan,             nan,             nan],
         [            nan,             nan,             nan, ...,
                      nan,             nan,             nan]],

        [[            nan,             nan,             nan, ...,
                      nan,             nan,             nan],
         [            nan,             nan,             nan, ...,
                      nan,             nan,             nan],
         [            nan,             nan,             nan, ...,
                      nan,             nan,             nan],
         ...,
         [-3.61188907e-04,  1.13505412e-05,  2.51400450e-04, ...,
           1.32970666e-05, -8.72688922e-04,  2.53200826e-04],
         [-4.17980340e-04, -2.25260483e-04, -5.59587289e-04, ...,
          -2.90739620e-05,  1.28007870e-03,  1.62103413e-04],
         [ 1.32660655e-03,  6.83768698e-05,  4.30072660e-04, ...,
           1.49505872e-04, -2.02690978e-03, -2.44386869e-04]]]])
Coordinates:
  * test_fold_ind  (test_fold_ind) int64 40B 0 1 2 3 4
  * weights        (weights) object 1kB 'hits_0' 'hits_1' ... 'running_9'
  * cell_roi_id    (cell_roi_id) object 5kB 'VISp_7_0000' ... 'VISp_6_0047'
  * model          (model) object 256B 'Full' ... 'single-behavioral'

In [85]:
np.where(np.isnan(W_cv_par))

(array([0, 0, 0, ..., 4, 4, 4]),
 array([ 1,  1,  1, ..., 31, 31, 31]),
 array([ 89,  89,  89, ..., 128, 128, 128]),
 array([  0,   1,   2, ..., 632, 633, 634]))

In [88]:
a = np.array([np.nan, np.nan, np.nan])
np.nanmean(a)

/tmp/ipykernel_15714/1537905765.py:2: RuntimeWarning: Mean of empty slice
  np.nanmean(a)


nan

In [94]:
W_cv_par.dims

('test_fold_ind', 'model', 'weights', 'cell_roi_id')

In [96]:
for dim in W_cv_par.dims:
    num_nan = len(np.where(np.isnan(W_cv_par.mean(dim=dim, skipna=True)))[0])
    print(f'NaN values averaged across {dim}: {num_nan}')

NaN values averaged across test_fold_ind: 1429385
NaN values averaged across model: 0
NaN values averaged across weights: 0
NaN values averaged across cell_roi_id: 11255


In [186]:
np.where(np.isnan(W_fold_par))

(array([ 1,  1,  1, ..., 31, 31, 31]),
 array([ 89,  89,  89, ..., 128, 128, 128]),
 array([  0,   1,   2, ..., 632, 633, 634]))

In [190]:
np.where(np.isnan(W_cv_par.isel(test_fold_ind=4)))

(array([ 1,  1,  1, ..., 31, 31, 31]),
 array([ 89,  89,  89, ..., 128, 128, 128]),
 array([  0,   1,   2, ..., 632, 633, 634]))

In [100]:
len(np.unique(np.where(np.isnan(W_cv))[3]))

635

In [116]:
ci = 20
w_cell = W_cv.isel(cell_roi_id = ci)
print(len(np.where(np.isnan(w_cell))[0]))
print(len(w_cell.values.flatten()))


11255
24160


In [120]:
num_nan = []
for cri in W_cv.cell_roi_id.values:
    w_cell = W_cv.sel(cell_roi_id=cri)
    num_nan.append(len(np.where(np.isnan(w_cell))[0]))
print(np.unique(num_nan))

[11255]


In [133]:
nan_inds = []
for cri in W_cv.cell_roi_id.values:
    w_cell = W_cv.sel(cell_roi_id=cri)
    nan_inds.append(np.array(np.where(np.isnan(w_cell))))
print(np.all([np.array_equal(nan_inds[0], nan_inds[i]) for i in range(1,len(nan_inds))]))

True


### When looked cell by cell, the number of nans are the same
- Indices are the same too. 
### What is the pattern?

In [136]:
w_cell.dims

('test_fold_ind', 'model', 'weights')

In [134]:
nan_inds[0]

array([[  0,   0,   0, ...,   4,   4,   4],
       [  1,   2,   2, ...,  31,  31,  31],
       [ 89,   0,   1, ..., 126, 127, 128]])

In [137]:
w_cell.model

<xarray.DataArray 'model' (model: 32)> Size: 256B
array(['Full', 'intercept', 'hits', 'misses', 'running', 'licks', 'im061',
       'im062', 'im063', 'im065', 'im066', 'im069', 'im077', 'im085',
       'all-images', 'task', 'behavioral', 'single-hits', 'single-misses',
       'single-running', 'single-licks', 'single-im061', 'single-im062',
       'single-im063', 'single-im065', 'single-im066', 'single-im069',
       'single-im077', 'single-im085', 'single-all-images', 'single-task',
       'single-behavioral'], dtype=object)
Coordinates:
    cell_roi_id  <U11 44B 'VISp_6_0047'
  * model        (model) object 256B 'Full' 'intercept' ... 'single-behavioral'

In [139]:
w_cell.weights[89]

<xarray.DataArray 'weights' ()> Size: 44B
array('intercept_0', dtype='<U11')
Coordinates:
    weights      <U11 44B 'intercept_0'
    cell_roi_id  <U11 44B 'VISp_6_0047'

In [141]:
test_fold_ind = 0
train_frames, test_frames, nested_fold_inds = gft.get_train_test_inds(test_fold_ind, fit_params, stratified_frames, cv_inds_stratified)
        
X_train_outer = X_trim[train_frames, :]
X_test_outer = X_trim[test_frames, :]
y_train_outer = response_trim_filtered[train_frames, :]
y_test_outer = response_trim_filtered[test_frames, :]

In [149]:
np.where(np.isnan(y_test_outer))

(array([], dtype=int64), array([], dtype=int64))

In [164]:
W_fold_par = W_fold.copy()

In [165]:
lambdas_fold, W_fold, var_ratio_train_fold, var_ratio_test_fold, vr_test_train_ratio_fold = \
            gft.collect_fold_results(run_params, fit_params, X_train_outer, X_test_outer, y_train_outer, y_test_outer, nested_fold_inds,
                                 parallel=False, num_cores=None)

KeyboardInterrupt: 

In [166]:
fold_inds = np.where(nan_inds[0][0, :] == test_fold_ind)[0]
nan_inds[0][:,fold_inds]

array([[  0,   0,   0, ...,   0,   0,   0],
       [  1,   2,   2, ...,  31,  31,  31],
       [ 89,   0,   1, ..., 126, 127, 128]])

In [182]:
W_fold_nan_inds = np.asarray(np.where(np.isnan(W_fold_par)))
fold_nan_inds_per_cell = []
for cri in range(len(W_fold_par.cell_roi_id)):
    W_fold_cell_ind = np.where(W_fold_nan_inds[2,:] == cri)[0]
    fold_nan_inds_per_cell.append(W_fold_nan_inds[:2,W_fold_cell_ind])
print(np.all([np.array_equal(fold_nan_inds_per_cell[0], 
                             fold_nan_inds_per_cell[i]) for i in range(1,len(nan_inds))]))

True


In [193]:
nan_inds_per_fold_per_cell = []
for fi in range(len(W_cv.test_fold_ind)):
    W_fold = W_cv.isel(test_fold_ind=fi)
    for ci in range(len(W_cv.cell_roi_id)):
        W_fold_cell = W_fold.isel(cell_roi_id=ci)
        nan_inds_per_fold_per_cell.append(np.array(np.where(np.isnan(W_fold_cell))))
print(np.all([np.array_equal(nan_inds_per_fold_per_cell[0], 
                             nan_inds_per_fold_per_cell[i]) for i in range(1,len(nan_inds))]))

True


In [194]:
nan_inds_per_fold_per_cell = nan_inds_per_fold_per_cell[0]
nan_inds_per_fold_per_cell

array([[  1,   2,   2, ...,  31,  31,  31],
       [ 89,   0,   1, ..., 126, 127, 128]])

In [226]:
model_ind = 15
model = W_cv_par.model.values[model_ind]
print(f'{model}')
nan_weight_inds = nan_inds_per_fold_per_cell[1, 
                                             np.where(nan_inds_per_fold_per_cell[0,:]==model_ind)[0]]
print(f'{W_cv_par.weights[nan_weight_inds]}')

task
<xarray.DataArray 'weights' (weights: 34)> Size: 272B
array(['hits_0', 'hits_1', 'hits_10', 'hits_11', 'hits_12', 'hits_13',
       'hits_14', 'hits_15', 'hits_16', 'hits_2', 'hits_3', 'hits_4', 'hits_5',
       'hits_6', 'hits_7', 'hits_8', 'hits_9', 'misses_0', 'misses_1',
       'misses_10', 'misses_11', 'misses_12', 'misses_13', 'misses_14',
       'misses_15', 'misses_16', 'misses_2', 'misses_3', 'misses_4',
       'misses_5', 'misses_6', 'misses_7', 'misses_8', 'misses_9'],
      dtype=object)
Coordinates:
  * weights  (weights) object 272B 'hits_0' 'hits_1' ... 'misses_8' 'misses_9'


In [239]:
weights = W_cv.weights.values
model_labels = W_cv.model.values

assert set(model_labels) == set(run_params['dropouts'].keys())
for model_label in model_labels:
# model_label = model_labels[3]
# print(model_label)
    model_kernels = run_params['dropouts'][model_label]['kernels']    
    run_weights = [w for w in weights if np.any([mk in w for mk in model_kernels])]
    dropped_weights = [w for w in weights if np.all([mk not in w for mk in model_kernels])]

    # are all run_weights finite?
    assert np.all(np.isfinite(W_cv.sel(weights=run_weights, model=model_label)))
    # are all dropped_weights nan?
    assert np.all(np.isnan(W_cv.sel(weights=dropped_weights, model=model_label)))

In [234]:
dropped_weights

['misses_0',
 'misses_1',
 'misses_10',
 'misses_11',
 'misses_12',
 'misses_13',
 'misses_14',
 'misses_15',
 'misses_16',
 'misses_2',
 'misses_3',
 'misses_4',
 'misses_5',
 'misses_6',
 'misses_7',
 'misses_8',
 'misses_9']

In [233]:
run_weights

['hits_0',
 'hits_1',
 'hits_10',
 'hits_11',
 'hits_12',
 'hits_13',
 'hits_14',
 'hits_15',
 'hits_16',
 'hits_2',
 'hits_3',
 'hits_4',
 'hits_5',
 'hits_6',
 'hits_7',
 'hits_8',
 'hits_9',
 'im061_0',
 'im061_1',
 'im061_2',
 'im061_3',
 'im061_4',
 'im061_5',
 'im061_6',
 'im061_7',
 'im061_8',
 'im062_0',
 'im062_1',
 'im062_2',
 'im062_3',
 'im062_4',
 'im062_5',
 'im062_6',
 'im062_7',
 'im062_8',
 'im063_0',
 'im063_1',
 'im063_2',
 'im063_3',
 'im063_4',
 'im063_5',
 'im063_6',
 'im063_7',
 'im063_8',
 'im065_0',
 'im065_1',
 'im065_2',
 'im065_3',
 'im065_4',
 'im065_5',
 'im065_6',
 'im065_7',
 'im065_8',
 'im066_0',
 'im066_1',
 'im066_2',
 'im066_3',
 'im066_4',
 'im066_5',
 'im066_6',
 'im066_7',
 'im066_8',
 'im069_0',
 'im069_1',
 'im069_2',
 'im069_3',
 'im069_4',
 'im069_5',
 'im069_6',
 'im069_7',
 'im069_8',
 'im077_0',
 'im077_1',
 'im077_2',
 'im077_3',
 'im077_4',
 'im077_5',
 'im077_6',
 'im077_7',
 'im077_8',
 'im085_0',
 'im085_1',
 'im085_2',
 'im085_3',
 '

In [227]:
run_params['dropouts'][model]['kernels']

['intercept',
 'running',
 'licks',
 'im061',
 'im062',
 'im063',
 'im065',
 'im066',
 'im069',
 'im077',
 'im085']

In [201]:
W_cv_par.weights

<xarray.DataArray 'weights' (weights: 151)> Size: 1kB
array(['hits_0', 'hits_1', 'hits_10', 'hits_11', 'hits_12', 'hits_13',
       'hits_14', 'hits_15', 'hits_16', 'hits_2', 'hits_3', 'hits_4', 'hits_5',
       'hits_6', 'hits_7', 'hits_8', 'hits_9', 'im061_0', 'im061_1', 'im061_2',
       'im061_3', 'im061_4', 'im061_5', 'im061_6', 'im061_7', 'im061_8',
       'im062_0', 'im062_1', 'im062_2', 'im062_3', 'im062_4', 'im062_5',
       'im062_6', 'im062_7', 'im062_8', 'im063_0', 'im063_1', 'im063_2',
       'im063_3', 'im063_4', 'im063_5', 'im063_6', 'im063_7', 'im063_8',
       'im065_0', 'im065_1', 'im065_2', 'im065_3', 'im065_4', 'im065_5',
       'im065_6', 'im065_7', 'im065_8', 'im066_0', 'im066_1', 'im066_2',
       'im066_3', 'im066_4', 'im066_5', 'im066_6', 'im066_7', 'im066_8',
       'im069_0', 'im069_1', 'im069_2', 'im069_3', 'im069_4', 'im069_5',
       'im069_6', 'im069_7', 'im069_8', 'im077_0', 'im077_1', 'im077_2',
       'im077_3', 'im077_4', 'im077_5', 'im077_6', 'im077_7', 'im077_8',
       'im085_0', 'im085_1', 'im085_2', 'im085_3', 'im085_4', 'im085_5',
       'im085_6', 'im085_7', 'im085_8', 'intercept_0', 'licks_-1', 'licks_-10',
       'licks_-11', 'licks_-2', 'licks_-3', 'licks_-4', 'licks_-5', 'licks_-6',
       'licks_-7', 'licks_-8', 'licks_-9', 'licks_0', 'licks_1', 'licks_10',
       'licks_2', 'licks_3', 'licks_4', 'licks_5', 'licks_6', 'licks_7',
       'licks_8', 'licks_9', 'misses_0', 'misses_1', 'misses_10', 'misses_11',
       'misses_12', 'misses_13', 'misses_14', 'misses_15', 'misses_16',
       'misses_2', 'misses_3', 'misses_4', 'misses_5', 'misses_6', 'misses_7',
       'misses_8', 'misses_9', 'running_-1', 'running_-10', 'running_-11',
       'running_-2', 'running_-3', 'running_-4', 'running_-5', 'running_-6',
       'running_-7', 'running_-8', 'running_-9', 'running_0', 'running_1',
       'running_10', 'running_2', 'running_3', 'running_4', 'running_5',
       'running_6', 'running_7', 'running_8', 'running_9'], dtype=object)
Coordinates:
  * weights  (weights) object 1kB 'hits_0' 'hits_1' ... 'running_8' 'running_9'

In [195]:
W_cv_par.model

<xarray.DataArray 'model' (model: 32)> Size: 256B
array(['Full', 'intercept', 'hits', 'misses', 'running', 'licks', 'im061',
       'im062', 'im063', 'im065', 'im066', 'im069', 'im077', 'im085',
       'all-images', 'task', 'behavioral', 'single-hits', 'single-misses',
       'single-running', 'single-licks', 'single-im061', 'single-im062',
       'single-im063', 'single-im065', 'single-im066', 'single-im069',
       'single-im077', 'single-im085', 'single-all-images', 'single-task',
       'single-behavioral'], dtype=object)
Coordinates:
  * model    (model) object 256B 'Full' 'intercept' ... 'single-behavioral'

In [158]:
for cri in W_fold_par.cell_roi_id:
    

array([0, 0, 0, ..., 4, 4, 4])

In [108]:
w_cell

<xarray.DataArray (test_fold_ind: 5, model: 32, weights: 151)> Size: 193kB
array([[[-3.21800146e-03,  4.66368199e-03, -2.68123246e-03, ...,
          6.47890033e-04,  2.61901640e-04,  1.32429972e-03],
        [-4.27402503e-03,  7.51206766e-03, -4.84691756e-03, ...,
          1.16651620e-03, -2.96712733e-04,  1.59463946e-03],
        [            nan,             nan,             nan, ...,
          8.62366149e-04,  1.05413902e-04,  1.41978927e-03],
        ...,
        [            nan,             nan,             nan, ...,
                     nan,             nan,             nan],
        [-8.71191403e-03,  7.72290245e-03,  5.57608396e-03, ...,
                     nan,             nan,             nan],
        [            nan,             nan,             nan, ...,
          1.29021544e-04,  5.56024505e-04,  8.63259915e-04]],

       [[-3.48537456e-03, -1.63028875e-05,  1.45912507e-02, ...,
          7.86437886e-04, -4.40579081e-04,  1.50283593e-03],
        [-4.61373354e-03,  6.73915490e-04,  1.99311197e-02, ...,
          1.04931919e-03, -8.94945009e-04,  1.72985643e-03],
        [            nan,             nan,             nan, ...,
          8.30269002e-04, -4.89948133e-04,  1.55224290e-03],
...
        [            nan,             nan,             nan, ...,
                     nan,             nan,             nan],
        [-6.89300027e-03,  6.18687965e-03,  2.98773774e-02, ...,
                     nan,             nan,             nan],
        [            nan,             nan,             nan, ...,
          4.41389971e-04, -7.43248214e-04,  1.69965389e-03]],

       [[-2.53339509e-03,  2.52018523e-03,  1.24086189e-02, ...,
         -5.19081064e-04, -3.24568848e-04,  1.52206477e-03],
        [-3.68558604e-03,  5.39767303e-03,  2.08113720e-02, ...,
         -3.43950513e-04, -7.28549656e-04,  1.75000269e-03],
        [            nan,             nan,             nan, ...,
         -5.20461167e-04, -3.36564800e-04,  1.56015823e-03],
        ...,
        [            nan,             nan,             nan, ...,
                     nan,             nan,             nan],
        [-7.72343594e-03,  4.48903751e-03,  3.00340462e-02, ...,
                     nan,             nan,             nan],
        [            nan,             nan,             nan, ...,
         -3.61188907e-04, -4.17980340e-04,  1.32660655e-03]]])
Coordinates:
  * test_fold_ind  (test_fold_ind) int64 40B 0 1 2 3 4
  * weights        (weights) object 1kB 'hits_0' 'hits_1' ... 'running_9'
    cell_roi_id    <U11 44B 'VISp_7_0000'
  * model          (model) object 256B 'Full' ... 'single-behavioral'

In [56]:
len(np.where(np.isnan(W_cv))[0])

7146925

In [63]:
len(np.unique(np.where(np.isnan(W_cv))[0]))

5

In [60]:
len(np.where(np.isnan(W_cv))[0]) /(5*32*151*635)

0.46585264900662254

In [ ]:
W_cv

<xarray.DataArray (test_fold_ind: 5, model: 32, weights: 151, cell_roi_id: 635)> Size: 123MB
array([[[[-3.21800146e-03,  1.75650484e-03,  1.94503345e-03, ...,
           2.96901546e-03,  1.36375263e-02,  5.26165594e-03],
         [ 4.66368199e-03,  1.00470187e-02,  5.05241984e-02, ...,
           1.73778314e-02,  2.16767038e-02,  1.74733317e-02],
         [-2.68123246e-03,  1.56128734e-03,  1.65456802e-02, ...,
          -1.59728474e-03, -1.94020901e-02, -5.01453608e-03],
         ...,
         [ 6.47890033e-04,  4.43339906e-04,  5.38367469e-04, ...,
          -2.29575859e-04,  1.40724789e-03, -1.12411516e-03],
         [ 2.61901640e-04, -3.65061820e-04,  1.17177388e-04, ...,
           6.67060688e-05,  8.59074741e-04,  8.84517873e-04],
         [ 1.32429972e-03,  2.56214715e-04,  7.36689530e-05, ...,
           1.06142872e-04, -2.92492354e-03, -6.46081711e-04]],

        [[-4.27402503e-03,  1.74545466e-03,  1.91891589e-03, ...,
           2.96211398e-03,  1.64256469e-02,  5.24649519e-03],
         [ 7.51206766e-03,  1.01064806e-02,  5.05944705e-02, ...,
           1.73980991e-02,  2.71727590e-02,  1.75399026e-02],
         [-4.84691756e-03,  1.49186409e-03,  1.64382400e-02, ...,
          -1.62692504e-03, -2.66077960e-02, -5.09704340e-03],
...
         [            nan,             nan,             nan, ...,
                      nan,             nan,             nan],
         [            nan,             nan,             nan, ...,
                      nan,             nan,             nan],
         [            nan,             nan,             nan, ...,
                      nan,             nan,             nan]],

        [[            nan,             nan,             nan, ...,
                      nan,             nan,             nan],
         [            nan,             nan,             nan, ...,
                      nan,             nan,             nan],
         [            nan,             nan,             nan, ...,
                      nan,             nan,             nan],
         ...,
         [-3.61188907e-04,  1.13505412e-05,  2.51400450e-04, ...,
           1.32970666e-05, -8.72688922e-04,  2.53200826e-04],
         [-4.17980340e-04, -2.25260483e-04, -5.59587289e-04, ...,
          -2.90739620e-05,  1.28007870e-03,  1.62103413e-04],
         [ 1.32660655e-03,  6.83768698e-05,  4.30072660e-04, ...,
           1.49505872e-04, -2.02690978e-03, -2.44386869e-04]]]])
Coordinates:
  * test_fold_ind  (test_fold_ind) int64 40B 0 1 2 3 4
  * weights        (weights) object 1kB 'hits_0' 'hits_1' ... 'running_9'
  * cell_roi_id    (cell_roi_id) object 5kB 'VISp_7_0000' ... 'VISp_6_0047'
  * model          (model) object 256B 'Full' ... 'single-behavioral'

# Test

In [64]:
load_fn = '/root/capsule/scratch/glm_results_v00_multiplane-ophys_721291_2024-05-16_08-57-00_filtered_events.npy'
results = np.load(load_fn, allow_pickle=True).item()

In [67]:
results['W_cv'].weights

<xarray.DataArray 'weights' (weights: 168)> Size: 1kB
array(['hits_0', 'hits_1', 'hits_10', 'hits_11', 'hits_12', 'hits_13',
       'hits_14', 'hits_15', 'hits_16', 'hits_2', 'hits_3', 'hits_4', 'hits_5',
       'hits_6', 'hits_7', 'hits_8', 'hits_9', 'im000_0', 'im000_1', 'im000_2',
       'im000_3', 'im000_4', 'im000_5', 'im000_6', 'im000_7', 'im000_8',
       'im031_0', 'im031_1', 'im031_2', 'im031_3', 'im031_4', 'im031_5',
       'im031_6', 'im031_7', 'im031_8', 'im035_0', 'im035_1', 'im035_2',
       'im035_3', 'im035_4', 'im035_5', 'im035_6', 'im035_7', 'im035_8',
       'im045_0', 'im045_1', 'im045_2', 'im045_3', 'im045_4', 'im045_5',
       'im045_6', 'im045_7', 'im045_8', 'im054_0', 'im054_1', 'im054_2',
       'im054_3', 'im054_4', 'im054_5', 'im054_6', 'im054_7', 'im054_8',
       'im073_0', 'im073_1', 'im073_2', 'im073_3', 'im073_4', 'im073_5',
       'im073_6', 'im073_7', 'im073_8', 'im075_0', 'im075_1', 'im075_2',
       'im075_3', 'im075_4', 'im075_5', 'im075_6', 'im075_7', 'im075_8',
       'im106_0', 'im106_1', 'im106_2', 'im106_3', 'im106_4', 'im106_5',
       'im106_6', 'im106_7', 'im106_8', 'intercept_0', 'licks_-1', 'licks_-10',
       'licks_-11', 'licks_-2', 'licks_-3', 'licks_-4', 'licks_-5', 'licks_-6',
       'licks_-7', 'licks_-8', 'licks_-9', 'licks_0', 'licks_1', 'licks_10',
       'licks_2', 'licks_3', 'licks_4', 'licks_5', 'licks_6', 'licks_7',
       'licks_8', 'licks_9', 'misses_0', 'misses_1', 'misses_10', 'misses_11',
       'misses_12', 'misses_13', 'misses_14', 'misses_15', 'misses_16',
       'misses_2', 'misses_3', 'misses_4', 'misses_5', 'misses_6', 'misses_7',
       'misses_8', 'misses_9', 'omissions_0', 'omissions_1', 'omissions_10',
       'omissions_11', 'omissions_12', 'omissions_13', 'omissions_14',
       'omissions_15', 'omissions_16', 'omissions_2', 'omissions_3',
       'omissions_4', 'omissions_5', 'omissions_6', 'omissions_7',
       'omissions_8', 'omissions_9', 'running_-1', 'running_-10',
       'running_-11', 'running_-2', 'running_-3', 'running_-4', 'running_-5',
       'running_-6', 'running_-7', 'running_-8', 'running_-9', 'running_0',
       'running_1', 'running_10', 'running_2', 'running_3', 'running_4',
       'running_5', 'running_6', 'running_7', 'running_8', 'running_9'],
      dtype=object)
Coordinates:
  * weights  (weights) object 1kB 'hits_0' 'hits_1' ... 'running_8' 'running_9'